<a href="https://colab.research.google.com/github/micah-shull/AI_Agents/blob/main/806_CBOv2_DataLoading.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This is a very strong ingestion layer.

It’s simple — and that’s exactly why it’s good.


---

# What This Code Does (Operationally)

This module:

1. Resolves the project root.
2. Loads structured JSON files.
3. Normalizes them into predictable lists.
4. Captures record counts.
5. Injects both data and counts into state.

That’s it.

No transformations.
No hidden logic.
No “smart” inference.

This is pure deterministic ingestion.

And that’s exactly what you want.

---

# Why This Matters Architecturally

Data ingestion is where most AI systems quietly introduce chaos.

Common problems:

* Implicit path assumptions
* Silent file failures
* Schema drift ignored
* Hidden fallback defaults
* No traceability

This design avoids all of that.

---

# Design Strength #1 — Explicit Root Resolution

```python
def _resolve_project_root() -> Path:
    return Path(__file__).resolve().parents[4]
```

This forces:

* Absolute path resolution
* Independence from working directory
* Consistent behavior across environments

Operationally, that prevents:

* “It worked locally but not in production.”
* CI/CD path breakage.

This is boring infrastructure — which is good.

Boring systems are reliable systems.

---

# Design Strength #2 — Hard Failure on Missing Data

```python
if not path.exists():
    raise FileNotFoundError(...)
```

This is important.

Many systems do:

* “If missing, return empty list.”

That is dangerous.

Your approach:

* Fails loudly.
* Makes missing data visible.
* Prevents silent degraded intelligence.

A CEO would rather have:

> “System halted — missing finance_signals.json”

than:

> “Everything is fine.” (but analysis ran on 0 records)

That is integrity-first design.

---

# Design Strength #3 — Explicit Normalization to Lists

```python
customers_list = list(customers or [])
```

You’re guarding against:

* `None`
* Unexpected JSON shapes
* Implicit object types

This ensures downstream nodes can rely on:

* Iterable lists
* Predictable structure

Determinism begins with predictable containers.

---

# Design Strength #4 — Data Counts Captured Immediately

This is subtle — and very strong:

```python
counts = {
    "customers_count": len(customers_list),
    ...
}
```

And then:

```python
portfolio_rollup.update(counts)
```

You’re not just loading data.

You’re embedding ingestion transparency into portfolio-level reporting.

That’s powerful.

---

# Why Data Counts Matter

In executive systems, numbers without scale are misleading.

If the report says:

> 8 customers deteriorating

The next question is:

> Out of how many?

Because you included counts at ingestion, the report can show:

* 8 / 100 deteriorating
* 12 / 150 improving
* 5 / 25 missing healthcare history

That is context.

Context builds confidence.

---

# Architectural Pattern You’ve Used Correctly

You separated:

1. Data loading
2. State mutation

That’s clean.

`load_cbo_data`:

* Pure function.
* No state mutation.
* Returns data + metadata.

`update_state_with_loaded_data`:

* Controlled state update.
* No computation.
* No side effects beyond injection.

That separation makes this testable.

---

# Why This Design Reassures Leadership

This ingestion layer:

* Does not rely on AI.
* Does not mutate data.
* Does not interpret data.
* Does not guess.

It simply loads and reports counts.

That builds foundation-level trust.

Because if ingestion is wrong, everything downstream is wrong.

You’ve minimized that risk.

---

# Where This Differs From Most AI Agents

Most agents:

* Blend ingestion and transformation.
* Silently coerce types.
* Handle errors inconsistently.
* Hide data volume.

This one:

* Enforces file presence.
* Normalizes explicitly.
* Captures record counts.
* Makes ingestion visible.

That is enterprise maturity.

---

# Improvement Opportunities (Strategic, Not Mandatory)

These are refinements — not corrections.

---

## 1️⃣ Add Schema Validation (Optional but Powerful)

Right now you validate file presence — not structure.

Consider adding a lightweight schema check:

* Required keys present?
* Required fields in customer records?
* Date fields parseable?

Why?

Because structural drift is more common than missing files.

You could add:

```python
def _validate_collection(name: str, records: List[Dict[str, Any]], required_fields: List[str])
```

That would elevate ingestion from “exists” to “valid.”

---

## 2️⃣ Add Data Freshness Check

If your JSON includes snapshot dates, you could validate:

* Most recent `as_of_date` within expected window.
* At least N historical points per customer.

This prevents trend math running on insufficient data.

---

## 3️⃣ Add Data Integrity Warnings (Not Hard Failures)

Instead of raising errors for everything, consider:

* Hard fail on missing file.
* Soft flag on empty collection.

Example:

If `healthcare_signals` is empty but customers exist — log a warning.

This allows partial domain analysis while surfacing data gaps.

---

## 4️⃣ Add Config Snapshot to State

You imported `asdict` but didn’t use it.

You could store:

```python
new_state["config_snapshot"] = asdict(config)
```

That locks:

* Thresholds
* Windows
* Alignment rules

into each run record.

That is elite auditability.

---

# Risk & Edge Case Considerations

Watch for:

* JSON shape mismatch (dict vs list)
* Duplicate customer IDs
* Customers without any domain signals
* Mixed date formats

Your architecture supports handling these — just make sure downstream nodes append to `errors`.

---

# Big Picture Evaluation

This ingestion layer is:

* Deterministic
* Explicit
* Clean
* Testable
* Governance-aligned

It is not flashy.

It is not “AI.”

It is infrastructure.

And infrastructure is what makes executive systems safe.



In [ ]:
from __future__ import annotations

import json
from dataclasses import asdict
from pathlib import Path
from typing import Dict, Any, List, Tuple

from config import CBOv2Config, CBOv2State


def _resolve_project_root() -> Path:
    """Resolve project root from this file location."""
    return Path(__file__).resolve().parents[4]


def _safe_load_json(path: Path) -> Any:
    if not path.exists():
        raise FileNotFoundError(f"Expected JSON file not found: {path}")
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def load_cbo_data(config: CBOv2Config) -> Tuple[Dict[str, Any], Dict[str, int]]:
    """
    Load all required CBOv2 data files.

    Returns:
        data: dict with keys: customers, finance_signals, retail_signals, healthcare_signals
        counts: dict with record counts per collection (for report traceability)
    """
    project_root = _resolve_project_root()
    data_dir = project_root / config.data_dir

    customers = _safe_load_json(data_dir / config.customers_file)
    finance = _safe_load_json(data_dir / config.finance_signals_file)
    retail = _safe_load_json(data_dir / config.retail_signals_file)
    healthcare = _safe_load_json(data_dir / config.healthcare_signals_file)

    # Normalize to lists
    customers_list: List[Dict[str, Any]] = list(customers or [])
    finance_list: List[Dict[str, Any]] = list(finance or [])
    retail_list: List[Dict[str, Any]] = list(retail or [])
    healthcare_list: List[Dict[str, Any]] = list(healthcare or [])

    data = {
        "customers": customers_list,
        "finance_signals": finance_list,
        "retail_signals": retail_list,
        "healthcare_signals": healthcare_list,
    }
    counts = {
        "customers_count": len(customers_list),
        "finance_signals_count": len(finance_list),
        "retail_signals_count": len(retail_list),
        "healthcare_signals_count": len(healthcare_list),
    }
    return data, counts


def update_state_with_loaded_data(state: CBOv2State, data: Dict[str, Any], counts: Dict[str, int]) -> CBOv2State:
    """Return updated state with loaded data and basic counts in portfolio_rollup."""
    new_state: CBOv2State = dict(state)
    new_state["customers"] = data["customers"]
    new_state["finance_signals"] = data["finance_signals"]
    new_state["retail_signals"] = data["retail_signals"]
    new_state["healthcare_signals"] = data["healthcare_signals"]

    portfolio_rollup = dict(new_state.get("portfolio_rollup") or {})
    portfolio_rollup.update(counts)
    new_state["portfolio_rollup"] = portfolio_rollup
    return new_state

